In [ ]:
# Sentinel-1 Crown-Level SAR Prototype

This notebook builds a parallel Sentinel-1 crown-level extraction workflow for the same 20 sampled crowns used in the Sentinel-2 prototype.

What this notebook does:
- Reuses the 20-crown sampled GeoJSON from the Sentinel-2 run.
- Queries Planetary Computer Sentinel-1 collections over the same AOI and years.
- Filters to a consistent orbit configuration where possible.
- Extracts per-crown VV and VH backscatter plus simple SAR features.
- Saves plots and long outputs to disk instead of rendering them inline.
- Compares whether per-crown Sentinel-1 features show label separation for deciduous vs evergreen/other crowns.
- Fuses Sentinel-1 crown-level features with the existing Sentinel-2 feature table for downstream modelling.

The notebook keeps inline output short. Inspect the saved CSV and PNG artifacts in the run folder for details.

hi


In [2]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Iterable
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from pystac_client import Client
import planetary_computer as pc
import rasterio
from rasterio import features
from rasterio.enums import Resampling
from pyproj import Transformer

warnings.filterwarnings('ignore', category=UserWarning)

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'output').exists() and (candidate / 'src').exists():
            return candidate
        if (candidate / 'Readme.md').exists() or (candidate / 'README.md').exists():
            return candidate
    return start

REPO_ROOT = find_repo_root(Path.cwd())
OUTPUT_ROOT = REPO_ROOT / 'output'
SOURCE_RUN_DIR = OUTPUT_ROOT / 'sat_data' / 'sit_scl_masked_6May26'
RUN_DIR = OUTPUT_ROOT / 'sat_data' / 'sit_sentinel1_6May26'
PLOTS_DIR = RUN_DIR / 'plots'
TABLES_DIR = RUN_DIR / 'tables'
META_DIR = RUN_DIR / 'meta'
for folder in [RUN_DIR, PLOTS_DIR, TABLES_DIR, META_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

STAC_API = 'https://planetarycomputer.microsoft.com/api/stac/v1'
COLLECTION_CANDIDATES = ['sentinel-1-rtc', 'sentinel-1-grd']
YEARS = [2022, 2023, 2024]
START_MM_DD = '01-01'
END_MM_DD = '12-31'
ONE_ITEM_PER_DATE = True
ALL_TOUCHED = True
PAD_PIXELS = 20
MIN_VALID_PIXELS = 1

SAMPLED_CROWNS_PATH = SOURCE_RUN_DIR / 'crowns_filtered_sample.geojson'
S2_FEATURES_PATH = SOURCE_RUN_DIR / 'tables' / 'crown_level_model_features.csv'
if not SAMPLED_CROWNS_PATH.exists():
    raise FileNotFoundError(f'Missing sampled crowns file: {SAMPLED_CROWNS_PATH}')
if not S2_FEATURES_PATH.exists():
    raise FileNotFoundError(f'Missing Sentinel-2 feature table: {S2_FEATURES_PATH}')

crowns = gpd.read_file(SAMPLED_CROWNS_PATH).copy()
crowns['chain_id'] = pd.to_numeric(crowns['chain_id'], errors='coerce')
crowns = crowns.dropna(subset=['chain_id']).copy()
crowns['chain_id'] = crowns['chain_id'].astype(int)
crowns = crowns[crowns.geometry.notnull() & ~crowns.geometry.is_empty].copy().reset_index(drop=True)

bounds = crowns.total_bounds
aoi = gpd.GeoDataFrame({'name': ['aoi']}, geometry=[crowns.unary_union.envelope], crs=crowns.crs)
aoi_wgs84 = aoi.to_crs('EPSG:4326')
bbox_wgs84 = [float(x) for x in aoi_wgs84.total_bounds]
centroid_wgs84 = aoi_wgs84.geometry.iloc[0].centroid

s2_features = pd.read_csv(S2_FEATURES_PATH).copy()
if 'chain_id' in s2_features.columns:
    s2_features['chain_id'] = pd.to_numeric(s2_features['chain_id'], errors='coerce').astype('Int64')
    s2_features = s2_features.dropna(subset=['chain_id']).copy()
    s2_features['chain_id'] = s2_features['chain_id'].astype(int)

run_meta = {
    'sampled_crowns_path': str(SAMPLED_CROWNS_PATH),
    'sentinel2_features_path': str(S2_FEATURES_PATH),
    'n_sampled_crowns': int(len(crowns)),
    'crowns_crs': str(crowns.crs),
    'bbox_wgs84': bbox_wgs84,
}
(META_DIR / 'run_meta.json').write_text(json.dumps(run_meta, indent=2), encoding='utf-8')

print('RUN_DIR:', RUN_DIR)
print('Sampled crowns:', len(crowns))
print('BBox WGS84:', bbox_wgs84)
print('Sentinel-2 feature rows:', len(s2_features))

RUN_DIR: /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_sentinel1_6May26
Sampled crowns: 20
BBox WGS84: [77.19065588450114, 28.544711067907027, 77.19232187419638, 28.546334350355384]
Sentinel-2 feature rows: 12


/var/folders/cx/8v11zwh97cxgdgt_ph13_56r0000gn/T/ipykernel_20912/2406966842.py:67: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  aoi = gpd.GeoDataFrame({'name': ['aoi']}, geometry=[crowns.unary_union.envelope], crs=crowns.crs)


In [3]:
catalog = Client.open(STAC_API)

def item_date_str(item) -> str | None:
    return item.datetime.strftime('%Y-%m-%d') if item.datetime else None

def postprocess_items(items: list, collection_name: str) -> pd.DataFrame:
    rows = []
    for item in items:
        dt = item_date_str(item)
        if dt is None:
            continue
        props = item.properties
        rows.append({
            'collection': collection_name,
            'id': item.id,
            'datetime': dt,
            'orbit_state': props.get('sat:orbit_state'),
            'relative_orbit': props.get('sat:relative_orbit'),
            'platform': props.get('platform'),
            'sar_mode': props.get('sar:instrument_mode'),
            'polarizations': ','.join(props.get('sar:polarizations', [])) if isinstance(props.get('sar:polarizations'), list) else props.get('sar:polarizations'),
            'epsg': props.get('proj:epsg'),
        })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df['date'] = pd.to_datetime(df['datetime'], errors='coerce')
    df = df.dropna(subset=['date']).copy()
    df['year'] = df['date'].dt.year.astype(int)
    iso = df['date'].dt.isocalendar()
    df['iso_year'] = iso.year.astype(int)
    df['iso_week'] = iso.week.astype(int)
    return df.sort_values('date').reset_index(drop=True)

selected_collection = None
items_df_raw = pd.DataFrame()
items_lookup = {}

for collection_name in COLLECTION_CANDIDATES:
    items = []
    for year in YEARS:
        start_date = f'{year}-{START_MM_DD}'
        end_date = f'{year}-{END_MM_DD}'
        search = catalog.search(
            collections=[collection_name],
            bbox=bbox_wgs84,
            datetime=f'{start_date}/{end_date}',
        )
        items.extend(list(search.get_items()))
    candidate_df = postprocess_items(items, collection_name)
    if candidate_df.empty:
        continue

    if 'polarizations' in candidate_df.columns:
        candidate_df = candidate_df[candidate_df['polarizations'].astype(str).str.contains('VV', na=False)]
        candidate_df = candidate_df[candidate_df['polarizations'].astype(str).str.contains('VH', na=False)]

    if 'sar_mode' in candidate_df.columns:
        candidate_df = candidate_df[candidate_df['sar_mode'].fillna('IW') == 'IW']

    if candidate_df.empty:
        continue

    orbit_counts = (
        candidate_df.groupby(['orbit_state', 'relative_orbit'], dropna=False)
        .size()
        .reset_index(name='n_items')
        .sort_values(['n_items', 'orbit_state', 'relative_orbit'], ascending=[False, True, True])
        .reset_index(drop=True)
    )
    chosen_orbit = orbit_counts.iloc[0]
    candidate_df = candidate_df[
        (candidate_df['orbit_state'] == chosen_orbit['orbit_state'])
        & (candidate_df['relative_orbit'] == chosen_orbit['relative_orbit'])
    ].copy()

    if ONE_ITEM_PER_DATE:
        candidate_df = (
            candidate_df.sort_values(['date', 'id'])
            .groupby('date', as_index=False)
            .first()
        )

    selected_collection = collection_name
    items_df_raw = candidate_df.sort_values('date').reset_index(drop=True)
    items_lookup = {item.id: item for item in items if item.id in set(items_df_raw['id'].tolist())}
    orbit_counts.to_csv(TABLES_DIR / f'{collection_name}_orbit_counts.csv', index=False)
    break

if selected_collection is None or items_df_raw.empty:
    raise RuntimeError('No usable Sentinel-1 collection/items found for this AOI.')

items_df_raw.to_csv(TABLES_DIR / 'sentinel1_items_filtered.csv', index=False)

print('Selected Sentinel-1 collection:', selected_collection)
print('Filtered item count:', len(items_df_raw))
print('Years represented:', sorted(items_df_raw['year'].unique().tolist()))
print('Orbit state:', items_df_raw['orbit_state'].iloc[0])
print('Relative orbit:', items_df_raw['relative_orbit'].iloc[0])

/Users/hbot07/anaconda3/envs/detectree/lib/python3.10/site-packages/pystac_client/item_search.py:925: FutureWarning: get_items() is deprecated, use items() instead
  warnings.warn(


Selected Sentinel-1 collection: sentinel-1-rtc
Filtered item count: 84
Years represented: [2022, 2023, 2024]
Orbit state: ascending
Relative orbit: 27


In [4]:
def signed_href(item, preferred_keys: list[str]) -> str:
    for key in preferred_keys:
        asset = item.assets.get(key)
        if asset is not None:
            return pc.sign(asset.href)
    raise KeyError(f'Missing any of assets {preferred_keys} on item {item.id}')

def infer_item_crs(item) -> str:
    epsg = item.properties.get('proj:epsg')
    if epsg is not None:
        return f'EPSG:{int(epsg)}'
    if crowns.crs is None:
        raise RuntimeError('Neither item nor crowns have a CRS.')
    return str(crowns.crs)

def aoi_bounds_in_crs(crs_str: str) -> tuple[float, float, float, float]:
    minlon, minlat, maxlon, maxlat = bbox_wgs84
    transformer = Transformer.from_crs('EPSG:4326', crs_str, always_xy=True)
    xs, ys = transformer.transform(
        [minlon, minlon, maxlon, maxlon],
        [minlat, maxlat, minlat, maxlat],
    )
    return float(min(xs)), float(min(ys)), float(max(xs)), float(max(ys))

def read_item_bands(item):
    crs_str = infer_item_crs(item)
    bounds = aoi_bounds_in_crs(crs_str)
    href_vv = signed_href(item, ['vv', 'VV'])
    href_vh = signed_href(item, ['vh', 'VH'])
    with rasterio.open(href_vv) as src_vv, rasterio.open(href_vh) as src_vh:
        window = rasterio.windows.from_bounds(*bounds, transform=src_vv.transform)
        window = window.round_offsets().round_lengths()
        window = rasterio.windows.Window(
            col_off=max(0, window.col_off - PAD_PIXELS),
            row_off=max(0, window.row_off - PAD_PIXELS),
            width=min(src_vv.width - max(0, window.col_off - PAD_PIXELS), window.width + 2 * PAD_PIXELS),
            height=min(src_vv.height - max(0, window.row_off - PAD_PIXELS), window.height + 2 * PAD_PIXELS),
        )
        vv = src_vv.read(1, window=window, masked=True).astype('float32')
        vh = src_vh.read(1, window=window, masked=True).astype('float32')
        transform = rasterio.windows.transform(window, src_vv.transform)
    return vv, vh, transform, crs_str

def compute_sar_features(vv, vh):
    vv_arr = np.asarray(vv, dtype='float32')
    vh_arr = np.asarray(vh, dtype='float32')
    invalid = np.ma.getmaskarray(vv) | np.ma.getmaskarray(vh) | ~np.isfinite(vv_arr) | ~np.isfinite(vh_arr)
    vv_arr = np.where(invalid, np.nan, vv_arr)
    vh_arr = np.where(invalid, np.nan, vh_arr)

    vv_db = np.where(vv_arr > 0, 10.0 * np.log10(vv_arr), np.nan).astype('float32')
    vh_db = np.where(vh_arr > 0, 10.0 * np.log10(vh_arr), np.nan).astype('float32')
    ratio_linear = np.where((vv_arr > 0) & (vh_arr > 0), vh_arr / vv_arr, np.nan).astype('float32')
    ratio_db = (vh_db - vv_db).astype('float32')
    valid_mask = np.isfinite(vv_db) & np.isfinite(vh_db)
    return vv_db, vh_db, ratio_linear, ratio_db, valid_mask

def build_crown_masks(crowns_gdf: gpd.GeoDataFrame, transform, out_shape: tuple[int, int]):
    masks = {}
    for row in crowns_gdf.itertuples(index=False):
        mask_geom = features.geometry_mask(
            [row.geometry.__geo_interface__],
            out_shape=out_shape,
            transform=transform,
            invert=True,
            all_touched=ALL_TOUCHED,
        )
        masks[int(row.chain_id)] = {
            'mask': mask_geom,
            'total_pixels': int(mask_geom.sum()),
        }
    return masks

def zonal_stats_for_sar(vv_db, vh_db, ratio_linear, ratio_db, valid_mask, crown_masks):
    finite = np.isfinite(vv_db) & np.isfinite(vh_db) & valid_mask
    rows = []
    for chain_id, payload in crown_masks.items():
        mask_geom = payload['mask']
        total_pixels = int(payload['total_pixels'])
        sel = mask_geom & finite
        valid_pixels = int(sel.sum())
        if valid_pixels:
            rows.append({
                'chain_id': chain_id,
                'vv_db_mean': float(np.nanmean(vv_db[sel])),
                'vh_db_mean': float(np.nanmean(vh_db[sel])),
                'vh_vv_ratio_mean': float(np.nanmean(ratio_linear[sel])),
                'vh_minus_vv_db_mean': float(np.nanmean(ratio_db[sel])),
                'valid_pixels': valid_pixels,
                'total_pixels': total_pixels,
            })
        else:
            rows.append({
                'chain_id': chain_id,
                'vv_db_mean': np.nan,
                'vh_db_mean': np.nan,
                'vh_vv_ratio_mean': np.nan,
                'vh_minus_vv_db_mean': np.nan,
                'valid_pixels': valid_pixels,
                'total_pixels': total_pixels,
            })
    return pd.DataFrame(rows)

def save_sar_overlay(item, crowns_gdf: gpd.GeoDataFrame, out_path: Path) -> None:
    vv, vh, transform, crs_str = read_item_bands(item)
    vv_db, vh_db, ratio_linear, ratio_db, valid_mask = compute_sar_features(vv, vh)
    img = vv_db.copy()
    finite = np.isfinite(img)
    if finite.any():
        lo, hi = np.percentile(img[finite], [2, 98])
        img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    else:
        img = np.zeros_like(img, dtype='float32')

    fig, ax = plt.subplots(figsize=(9, 9), dpi=180)
    ax.imshow(img, cmap='gray')
    inv = ~transform
    crowns_plot = crowns_gdf.to_crs(crs_str)
    for geom in crowns_plot.geometry:
        if geom is None or geom.is_empty:
            continue
        parts = [geom] if geom.geom_type == 'Polygon' else list(getattr(geom, 'geoms', []))
        if not parts:
            continue
        poly = max(parts, key=lambda g: g.area)
        xs, ys = poly.exterior.xy
        coords = [inv * (x, y) for x, y in zip(xs, ys)]
        ax.plot([c[0] for c in coords], [c[1] for c in coords], color='cyan', linewidth=0.7, alpha=0.9)
    ax.set_title(f'Sentinel-1 VV backscatter with crowns\n{item.id}')
    ax.axis('off')
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    plt.close(fig)

print('Sentinel-1 helper functions ready.')

Sentinel-1 helper functions ready.


In [5]:
best_item = items_lookup[items_df_raw.sort_values('date').iloc[0]['id']]
save_sar_overlay(best_item, crowns, PLOTS_DIR / 'sentinel1_vv_overlay.png')

records = []
crown_masks = None
transform_signature = None

for idx, row in enumerate(items_df_raw.itertuples(index=False), start=1):
    item = items_lookup[row.id]
    vv, vh, transform, crs_str = read_item_bands(item)
    vv_db, vh_db, ratio_linear, ratio_db, valid_mask = compute_sar_features(vv, vh)

    current_signature = (crs_str, transform.a, transform.b, transform.c, transform.d, transform.e, transform.f, vv_db.shape)
    if crown_masks is None or current_signature != transform_signature:
        crowns_sar = crowns.to_crs(crs_str)
        crown_masks = build_crown_masks(crowns_sar, transform, vv_db.shape)
        transform_signature = current_signature

    stats_df = zonal_stats_for_sar(vv_db, vh_db, ratio_linear, ratio_db, valid_mask, crown_masks)
    for rec in stats_df.itertuples(index=False):
        records.append({
            'date': row.date,
            'datetime': row.datetime,
            'year': int(row.year),
            'iso_year': int(row.iso_year),
            'iso_week': int(row.iso_week),
            'month': int(pd.Timestamp(row.date).month),
            'item_id': row.id,
            'chain_id': int(rec.chain_id),
            'vv_db_mean': float(rec.vv_db_mean) if pd.notna(rec.vv_db_mean) else np.nan,
            'vh_db_mean': float(rec.vh_db_mean) if pd.notna(rec.vh_db_mean) else np.nan,
            'vh_vv_ratio_mean': float(rec.vh_vv_ratio_mean) if pd.notna(rec.vh_vv_ratio_mean) else np.nan,
            'vh_minus_vv_db_mean': float(rec.vh_minus_vv_db_mean) if pd.notna(rec.vh_minus_vv_db_mean) else np.nan,
            'valid_pixels': int(rec.valid_pixels),
            'total_pixels': int(rec.total_pixels),
            'valid_fraction': float(rec.valid_pixels / rec.total_pixels) if rec.total_pixels else np.nan,
            'is_valid': bool(rec.valid_pixels >= MIN_VALID_PIXELS),
        })

obs_df = pd.DataFrame.from_records(records)
obs_df['date'] = pd.to_datetime(obs_df['date'])
obs_df = obs_df.sort_values(['chain_id', 'date']).reset_index(drop=True)
label_cols = [col for col in ['chain_id', 'is_deciduous', 'deciduous_score', 'leaf_off_start_om', 'full_leaf_off_om', 'leaf_on_return_om'] if col in crowns.columns]
if label_cols:
    obs_df = obs_df.merge(crowns[label_cols].drop_duplicates('chain_id'), on='chain_id', how='left')

obs_df.to_csv(TABLES_DIR / 'sentinel1_observations_all.csv', index=False)
coverage = (
    obs_df.groupby('chain_id', as_index=False)
    .agg(n_dates=('date', 'nunique'), n_valid=('is_valid', 'sum'), median_valid_fraction=('valid_fraction', 'median'))
    .sort_values('n_valid', ascending=False)
    .reset_index(drop=True)
)
coverage.to_csv(TABLES_DIR / 'sentinel1_crown_coverage.csv', index=False)

print('Sentinel-1 observation rows:', len(obs_df))
print('Valid crown-date rows:', int(obs_df['is_valid'].sum()))
print('Saved observation tables to', TABLES_DIR)

Sentinel-1 observation rows: 1680
Valid crown-date rows: 1680
Saved observation tables to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_sentinel1_6May26/tables


In [8]:
valid_df = obs_df[obs_df['is_valid']].copy()

weekly_per_crown = (
    valid_df.groupby(['chain_id', 'iso_week'], as_index=False)
    .agg(
        vv_db_weekly_median=('vv_db_mean', 'median'),
        vh_db_weekly_median=('vh_db_mean', 'median'),
        vh_vv_ratio_weekly_median=('vh_vv_ratio_mean', 'median'),
        vh_minus_vv_db_weekly_median=('vh_minus_vv_db_mean', 'median'),
        n_obs=('vv_db_mean', 'size'),
    )
    .sort_values(['chain_id', 'iso_week'])
)
label_cols = [col for col in ['chain_id', 'is_deciduous', 'deciduous_score'] if col in crowns.columns]
if label_cols:
    weekly_per_crown = weekly_per_crown.merge(crowns[label_cols].drop_duplicates('chain_id'), on='chain_id', how='left')
weekly_per_crown.to_csv(TABLES_DIR / 'sentinel1_weekly_per_crown.csv', index=False)

fig, axes = plt.subplots(4, 5, figsize=(18, 12), dpi=180, sharex=True, sharey=True)
axes = axes.flatten()
top_chain_ids = coverage.head(20)['chain_id'].tolist()
for ax, chain_id in zip(axes, top_chain_ids):
    subset = weekly_per_crown[weekly_per_crown['chain_id'] == chain_id].sort_values('iso_week')
    if subset.empty:
        ax.axis('off')
        continue
    ax.plot(subset['iso_week'], subset['vv_db_weekly_median'], color='#1f4e79', linewidth=1.1, label='VV dB')
    ax.plot(subset['iso_week'], subset['vh_db_weekly_median'], color='#b22222', linewidth=1.0, alpha=0.9, label='VH dB')
    label_value = subset['is_deciduous'].dropna().iloc[0] if 'is_deciduous' in subset.columns and subset['is_deciduous'].notna().any() else None
    label_bool = True if str(label_value).strip().lower() == 'true' else False if str(label_value).strip().lower() == 'false' else None
    label = 'D' if label_bool is True else ('E/O' if label_bool is False else '?')
    ax.set_title(f'crown {chain_id} | {label}', fontsize=9)
    ax.set_xlim(1, 53)
    ax.grid(True, alpha=0.2)

for ax in axes[len(top_chain_ids):]:
    ax.axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False)
fig.suptitle('Per-crown weekly Sentinel-1 backscatter', fontsize=14)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(PLOTS_DIR / 'sentinel1_per_crown_weekly_grid.png', bbox_inches='tight')
plt.close(fig)

feature_rows = []
for chain_id, group in valid_df.groupby('chain_id'):
    row = {
        'chain_id': int(chain_id),
        'n_obs_s1': int(len(group)),
        'n_years_s1': int(group['year'].nunique()),
        'vv_db_mean_s1': float(group['vv_db_mean'].mean()),
        'vv_db_std_s1': float(group['vv_db_mean'].std(ddof=0)),
        'vv_db_amp_s1': float(group['vv_db_mean'].max() - group['vv_db_mean'].min()),
        'vh_db_mean_s1': float(group['vh_db_mean'].mean()),
        'vh_db_std_s1': float(group['vh_db_mean'].std(ddof=0)),
        'vh_db_amp_s1': float(group['vh_db_mean'].max() - group['vh_db_mean'].min()),
        'vh_vv_ratio_mean_s1': float(group['vh_vv_ratio_mean'].mean()),
        'vh_vv_ratio_std_s1': float(group['vh_vv_ratio_mean'].std(ddof=0)),
        'vh_minus_vv_db_mean_s1': float(group['vh_minus_vv_db_mean'].mean()),
        'vh_minus_vv_db_std_s1': float(group['vh_minus_vv_db_mean'].std(ddof=0)),
        'valid_fraction_median_s1': float(group['valid_fraction'].median()),
    }
    feature_rows.append(row)

s1_features = pd.DataFrame(feature_rows).sort_values('chain_id').reset_index(drop=True)
if label_cols:
    s1_features = s1_features.merge(crowns[label_cols].drop_duplicates('chain_id'), on='chain_id', how='left')
if 'is_deciduous' in s1_features.columns:
    s1_features['is_deciduous'] = s1_features['is_deciduous'].map(lambda x: True if str(x).strip().lower() == 'true' else False if str(x).strip().lower() == 'false' else np.nan)
s1_features.to_csv(TABLES_DIR / 'sentinel1_crown_level_features.csv', index=False)

separation_rows = []
label_ready = s1_features.dropna(subset=['is_deciduous']).copy()
for feature in ['vv_db_mean_s1', 'vv_db_amp_s1', 'vh_db_mean_s1', 'vh_db_amp_s1', 'vh_vv_ratio_mean_s1', 'vh_minus_vv_db_mean_s1']:
    group_true = pd.to_numeric(label_ready.loc[label_ready['is_deciduous'] == True, feature], errors='coerce').dropna()
    group_false = pd.to_numeric(label_ready.loc[label_ready['is_deciduous'] == False, feature], errors='coerce').dropna()
    if len(group_true) < 2 or len(group_false) < 2:
        continue
    mean_true = float(group_true.mean())
    mean_false = float(group_false.mean())
    var_true = float(group_true.var(ddof=0))
    var_false = float(group_false.var(ddof=0))
    pooled = float(np.sqrt((var_true + var_false) / 2.0)) if (var_true + var_false) > 0 else np.nan
    effect_size = float((mean_true - mean_false) / pooled) if pooled and np.isfinite(pooled) else np.nan
    separation_rows.append({
        'feature': feature,
        'mean_deciduous': mean_true,
        'mean_evergreen_other': mean_false,
        'cohen_d': effect_size,
        'n_deciduous': int(len(group_true)),
        'n_evergreen_other': int(len(group_false)),
    })

separation_df = pd.DataFrame(separation_rows)
if not separation_df.empty:
    separation_df = separation_df.sort_values('cohen_d', key=lambda s: s.abs(), ascending=False).reset_index(drop=True)
separation_df.to_csv(TABLES_DIR / 'sentinel1_label_separation.csv', index=False)

print('Weekly per-crown rows:', len(weekly_per_crown))
print('Crown-level S1 feature rows:', len(s1_features))
print('Label-ready crowns:', int(label_ready.shape[0]))
if not separation_df.empty:
    print('Top SAR separation features:')
    print(separation_df.head(6).to_string(index=False))
else:
    print('No label-separation table could be built with the current sample.')

Weekly per-crown rows: 1040
Crown-level S1 feature rows: 20
Label-ready crowns: 20
Top SAR separation features:
               feature  mean_deciduous  mean_evergreen_other   cohen_d  n_deciduous  n_evergreen_other
vh_minus_vv_db_mean_s1       -7.515633             -8.692269  0.803610           11                  9
   vh_vv_ratio_mean_s1        0.217020              0.167361  0.652652           11                  9
         vv_db_mean_s1       -3.171084             -1.274025 -0.637068           11                  9
         vh_db_mean_s1      -10.686717             -9.966294 -0.400291           11                  9
          vv_db_amp_s1        7.767270              8.264058 -0.329614           11                  9
          vh_db_amp_s1        7.368757              7.672948 -0.217017           11                  9


In [9]:
fused = s1_features.merge(s2_features, on='chain_id', how='inner', suffixes=('_s1', '_s2'))
if 'is_deciduous_s1' in fused.columns:
    fused['is_deciduous'] = fused['is_deciduous_s1']
elif 'is_deciduous_s2' in fused.columns:
    fused['is_deciduous'] = fused['is_deciduous_s2']
else:
    fused['is_deciduous'] = np.nan
fused.to_csv(TABLES_DIR / 'sentinel1_sentinel2_fused_features.csv', index=False)

scatter_source = fused.dropna(subset=['is_deciduous', 'vv_db_mean_s1', 'ndvi_mean']).copy()

fig, ax = plt.subplots(figsize=(6, 5), dpi=180)
for is_deciduous, color, label in [(True, '#b22222', 'Deciduous'), (False, '#1f4e79', 'Evergreen/other')]:
    subset = scatter_source[scatter_source['is_deciduous'] == is_deciduous]
    if subset.empty:
        continue
    ax.scatter(
        subset['ndvi_mean'],
        subset['vv_db_mean_s1'],
        s=50,
        alpha=0.85,
        color=color,
        label=label,
    )
    for row in subset.itertuples(index=False):
        ax.text(row.ndvi_mean, row.vv_db_mean_s1, str(int(row.chain_id)), fontsize=7, alpha=0.8)
ax.set_title('Sentinel-2 NDVI vs Sentinel-1 VV by crown')
ax.set_xlabel('Sentinel-2 NDVI mean')
ax.set_ylabel('Sentinel-1 VV mean (dB)')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 's1_s2_fused_scatter_ndvi_vv.png', bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 5), dpi=180)
for is_deciduous, color, label in [(True, '#b22222', 'Deciduous'), (False, '#1f4e79', 'Evergreen/other')]:
    subset = scatter_source[scatter_source['is_deciduous'] == is_deciduous]
    if subset.empty:
        continue
    ax.scatter(
        subset['ndvi_amp'],
        subset['vh_minus_vv_db_mean_s1'],
        s=50,
        alpha=0.85,
        color=color,
        label=label,
    )
    for row in subset.itertuples(index=False):
        ax.text(row.ndvi_amp, row.vh_minus_vv_db_mean_s1, str(int(row.chain_id)), fontsize=7, alpha=0.8)
ax.set_title('Sentinel-2 NDVI amplitude vs Sentinel-1 VH-VV')
ax.set_xlabel('Sentinel-2 NDVI amplitude')
ax.set_ylabel('Sentinel-1 VH - VV mean (dB)')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 's1_s2_fused_scatter_amp_ratio.png', bbox_inches='tight')
plt.close(fig)

print('Fused feature rows:', len(fused))
print('Scatter rows with labels:', len(scatter_source))
print('Saved fused features to', TABLES_DIR / 'sentinel1_sentinel2_fused_features.csv')
print('Saved fusion plots to', PLOTS_DIR)

Fused feature rows: 12
Scatter rows with labels: 12
Saved fused features to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_sentinel1_6May26/tables/sentinel1_sentinel2_fused_features.csv
Saved fusion plots to /Users/hbot07/VS Code/Drone-Phenology-Monitoring/output/sat_data/sit_sentinel1_6May26/plots
